# LOPOCV Experiments

In [1]:
import numpy as np
from pathlib import Path


def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


X, y, patient = load_dataset()

In [2]:
X[0]

array([[ -1.7578125,  -5.6640625,  -7.6171875, ...,   1.7578125,
         -0.1953125,   1.3671875],
       [ -8.0078125, -10.3515625, -10.3515625, ...,  10.3515625,
         13.0859375,  10.7421875],
       [-18.171875 , -16.609375 , -14.2578125, ..., -18.953125 ,
        -20.125    , -14.6484375],
       ...,
       [ 39.28125  ,  33.8125   ,  26.375    , ...,  -9.1796875,
        -12.3046875,  -6.0546875],
       [-22.078125 , -20.90625  , -24.421875 , ..., -18.5625   ,
        -21.296875 , -26.765625 ],
       [-46.6875   , -38.09375  , -26.765625 , ...,  -5.6640625,
          0.1953125,  -5.2734375]], shape=(21, 768), dtype=float32)

In [ ]:
"""
AsymSETNet: Asymmetric Squeeze-Excitation Temporal Network for EEG Seizure Detection
Implementation based on: "Enhancing Seizure Detection via Asymmetric Spatiotemporal
Feature Extraction" (Marihuan & Saavedra).

Pipeline (see Fig. 1 of the paper):
    Raw EEG -> N segments -> Spatial Module (Asymmetric Conv + SE) x2
             -> Global Average Pooling -> Temporal Module (stacked TCN)
             -> Classification Head (FC) -> Seizure / Normal
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------- #
# 1. Squeeze-and-Excitation block (2D, operates on conv feature maps)
# --------------------------------------------------------------------------- #
class SqueezeExcitation2d(nn.Module):
    """Channel-wise recalibration: squeeze (global avg pool) -> excitation
    (bottleneck MLP) -> scale (channel-wise multiplication). See Hu et al.,
    2019, as used between the convolutional layers of the Spatial Module."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        e = self.excite(s).view(b, c, 1, 1)
        return x * e


# --------------------------------------------------------------------------- #
# 2. Asymmetric convolutional block: Conv -> AvgPool -> SE -> Add & LayerNorm
# --------------------------------------------------------------------------- #
class AsymmetricConvBlock(nn.Module):
    """One stage of the Spatial Module (Section III-B).

    Applies an asymmetric (M x N) dilated convolution -> average pooling
    along the temporal axis -> Squeeze-and-Excitation recalibration -> a
    residual (skip) connection back to the block's input -> LayerNorm.

    A kernel height M equal to the current electrode/height dimension
    collapses that spatial axis to 1 in a single step (as in Spatial
    Module 1, kernel 22x4); a kernel height of 1 (Spatial Module 2,
    kernel 1x4) then only refines the temporal axis.
    """

    def __init__(self, in_channels, out_channels, kernel_size, dilation,
                 pool_size=(1, 2), se_reduction=16):
        super().__init__()
        kh, kw = kernel_size
        dh, dw = dilation

        # 'same'-style padding along the time axis; no padding along height
        # so that kh == current_height forces the electrode-dimension collapse
        # described in the paper (Eq. 1).
        pad_h = 0
        pad_w = ((kw - 1) * dw) // 2

        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=(kh, kw),
            dilation=(dh, dw), padding=(pad_h, pad_w),
        )
        self.pool = nn.AvgPool2d(kernel_size=pool_size, ceil_mode=True)
        self.se = SqueezeExcitation2d(out_channels, reduction=se_reduction)

        # Projection for the skip connection: mirrors the main branch's
        # height-collapsing behaviour (kernel/stride = kh along height) so the
        # residual add lines up in channels, height, and (after pooling) width.
        self.skip_proj = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(kh, 1),
                      stride=(kh, 1)),
            nn.AvgPool2d(kernel_size=pool_size, ceil_mode=True),
        )

        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        skip = self.skip_proj(x)

        y = self.conv(x)
        y = self.pool(y)
        y = self.se(y)

        # Defensive crop in case of pooling rounding mismatches.
        if y.shape[-1] != skip.shape[-1]:
            w = min(y.shape[-1], skip.shape[-1])
            y, skip = y[..., :w], skip[..., :w]
        if y.shape[-2] != skip.shape[-2]:
            h = min(y.shape[-2], skip.shape[-2])
            y, skip = y[..., :h, :], skip[..., :h, :]

        out = y + skip
        out = out.permute(0, 2, 3, 1)   # channels last for LayerNorm
        out = self.norm(out)
        out = out.permute(0, 3, 1, 2)
        return F.relu(out)


# --------------------------------------------------------------------------- #
# 3. Spatial Module: two stacked Asymmetric Conv (+SE) blocks
# --------------------------------------------------------------------------- #
class SpatialModule(nn.Module):
    """Extracts cross-channel (electrode) spatial features from one EEG
    segment. Kernel sizes follow Table I: (Cmax, 4) then (1, 4), both with
    dilation (1, 2)."""

    def __init__(self, n_channels, f1=16, f2=32, se_reduction=16):
        super().__init__()
        self.block1 = AsymmetricConvBlock(
            in_channels=1, out_channels=f1,
            kernel_size=(n_channels, 4), dilation=(1, 2),
            se_reduction=se_reduction,
        )
        self.block2 = AsymmetricConvBlock(
            in_channels=f1, out_channels=f2,
            kernel_size=(1, 4), dilation=(1, 2),
            se_reduction=se_reduction,
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        # x: (B, 1, C, T_segment)
        x = self.block1(x)          # -> (B, f1, 1, T')
        x = self.block2(x)          # -> (B, f2, 1, T'')
        return self.gap(x).flatten(1)   # -> (B, f2)


# --------------------------------------------------------------------------- #
# 4. Temporal Convolutional Network (Bai et al., 2018 style)
# --------------------------------------------------------------------------- #
class Chomp1d(nn.Module):
    """Trims the extra right-side padding used to keep dilated causal
    convolutions the same length as their input."""

    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """A single causal, dilated TCN residual block, with skip connection
    (Section III-C)."""

    def __init__(self, in_channels, out_channels, kernel_size, dilation,
                 dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.downsample = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else None
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TemporalConvNet(nn.Module):
    """Stack of TemporalBlocks with exponentially increasing dilation,
    expanding the receptive field across the sequence of N segments."""

    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i
            in_ch = num_inputs if i == 0 else num_channels[i - 1]
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size,
                                         dilation, dropout))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, C, N_segments)
        return self.network(x)


# --------------------------------------------------------------------------- #
# 5. Full AsymSETNet model
# --------------------------------------------------------------------------- #
class AsymSETNet(nn.Module):
    """End-to-end Asymmetric Squeeze-Excitation Temporal Network.

    Args:
        n_channels:   number of EEG electrode channels (Cmax), e.g. 22.
        n_segments:   number of temporal segments per window (N), e.g. 5.
        spatial_f1:   feature width of Spatial Module 1.
        spatial_f2:   feature width of Spatial Module 2.
        tcn_channels: output channel sizes for each of the (2) TCN layers.
        se_reduction: SE bottleneck reduction ratio (16 in the paper).
        dropout:      dropout used inside the TCN blocks.
    """

    def __init__(self, n_channels=22, n_segments=5, spatial_f1=16,
                 spatial_f2=32, tcn_channels=(32, 32), se_reduction=16,
                 dropout=0.2):
        super().__init__()
        self.n_segments = n_segments

        self.spatial_module = SpatialModule(
            n_channels=n_channels, f1=spatial_f1, f2=spatial_f2,
            se_reduction=se_reduction,
        )
        self.temporal_module = TemporalConvNet(
            num_inputs=spatial_f2, num_channels=list(tcn_channels),
            kernel_size=3, dropout=dropout,
        )
        self.classifier = nn.Linear(tcn_channels[-1], 1)

    def forward(self, x):
        """
        x: raw EEG window, shape (B, C, T_total)
        returns: raw logits for the seizure class, shape (B,)
        """
        b, c, t_total = x.shape
        n = self.n_segments
        assert t_total % n == 0, "T_total must be divisible by n_segments"
        t_seg = t_total // n

        # Segment the recording: (B, C, T_total) -> (B*N, 1, C, T_seg)
        x = x.view(b, c, n, t_seg).permute(0, 2, 1, 3).contiguous()
        x = x.view(b * n, 1, c, t_seg)

        # Spatial feature extraction per segment -> (B*N, f2)
        feats = self.spatial_module(x)
        feats = feats.view(b, n, -1).permute(0, 2, 1)   # (B, f2, N)

        # Temporal modeling across the N segments
        temporal_out = self.temporal_module(feats)      # (B, C_tcn, N)
        pooled = temporal_out[:, :, -1]                  # last segment embedding

        return self.classifier(pooled).squeeze(-1)       # (B,)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))


# --------------------------------------------------------------------------- #
# 6. Quick smoke test / training-loop sketch
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    torch.manual_seed(0)

    B, C, T_TOTAL = 8, 22, 5 * 256      # 5-second windows @ 256 Hz -> 1280 samples
    N_SEGMENTS = 5

    model = AsymSETNet(n_channels=C, n_segments=N_SEGMENTS)
    x = torch.randn(B, C, T_TOTAL)
    y = torch.randint(0, 2, (B,)).float()

    logits = model(x)
    loss_fn = nn.BCEWithLogitsLoss()
    loss = loss_fn(logits, y)
    loss.backward()  # verify gradients flow end-to-end

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Input shape:            {tuple(x.shape)}")
    print(f"Output (logits) shape:  {tuple(logits.shape)}")
    print(f"BCE loss:               {loss.item():.4f}")
    print(f"Trainable parameters:   {n_params:,}")

    # Example optimizer matching the paper's setup (Section IV-C)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    optimizer.step()
    print("One optimizer step completed successfully.")

Input shape:            (8, 22, 1280)
Output (logits) shape:  (8,)
BCE loss:               0.7382
Trainable parameters:   17,121
One optimizer step completed successfully.


In [4]:
fds

NameError: name 'fds' is not defined

In [ ]:
"""
Minimal Leave-One-Patient-Out Cross-Validation (LOPOCV) training script for
AsymSETNet on the pre-windowed CHB-MIT dataset.

Usage:
    python train_lopocv.py
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset



# --------------------------------------------------------------------------- #
# Train / evaluate a single fold
# --------------------------------------------------------------------------- #
def run_fold(X_train, y_train, X_test, y_test, n_segments=5,
             epochs=80, patience=20, batch_size=64, lr=1e-3,
             weight_decay=1e-4, device="cpu"):
    n_channels = X_train.shape[1]

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).float())
    test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).float())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = AsymSETNet(n_channels=n_channels, n_segments=n_segments).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    best_loss, best_state, no_improve = float("inf"), None, 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Validation loss (used here as the held-out fold, for early stopping)
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(loss_fn(model(xb), yb).item())
        val_loss = float(np.mean(val_losses))

        if val_loss < best_loss:
            best_loss, best_state, no_improve = val_loss, model.state_dict(), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)

    # Final evaluation: accuracy, sensitivity, specificity
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            probs = model.predict_proba(xb).cpu().numpy()
            all_preds.append((probs >= 0.5).astype(int))
            all_true.append(yb.numpy().astype(int))
    preds = np.concatenate(all_preds)
    trues = np.concatenate(all_true)

    tp = np.sum((preds == 1) & (trues == 1))
    tn = np.sum((preds == 0) & (trues == 0))
    fp = np.sum((preds == 1) & (trues == 0))
    fn = np.sum((preds == 0) & (trues == 1))

    acc = (tp + tn) / len(trues)
    sen = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return acc, sen, spec



device = "cuda" if torch.cuda.is_available() else "cpu"
X, y, patient = load_dataset()
results = {}
for pid in np.unique(patient):
    test_mask = patient == pid
    train_mask = ~test_mask
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    acc, sen, spec = run_fold(X_train, y_train, X_test, y_test, device=device, n_segments=3)
    results[pid] = (acc, sen, spec)
    print(f"Patient {pid}: Acc={acc:.4f}  Sen={sen:.4f}  Spec={spec:.4f}")
accs = [v[0] for v in results.values()]
sens = [v[1] for v in results.values()]
specs = [v[2] for v in results.values()]
print("\n=== LOPOCV summary ===")
print(f"Mean Acc:  {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print(f"Mean Sen:  {np.mean(sens):.4f} +/- {np.std(sens):.4f}")
print(f"Mean Spec: {np.mean(specs):.4f} +/- {np.std(specs):.4f}")

Patient 0: Acc=0.9281  Sen=1.0000  Spec=0.8562
Patient 1: Acc=0.8860  Sen=0.7895  Spec=0.9825
Patient 2: Acc=0.8817  Sen=0.8473  Spec=0.9160
Patient 3: Acc=0.8640  Sen=0.8960  Spec=0.8320
Patient 4: Acc=0.6000  Sen=0.9514  Spec=0.2486
Patient 5: Acc=0.5729  Sen=0.2500  Spec=0.8958
Patient 6: Acc=0.9206  Sen=0.9626  Spec=0.8785
Patient 7: Acc=0.6672  Sen=0.7115  Spec=0.6230
Patient 8: Acc=0.7000  Sen=0.9889  Spec=0.4111
Patient 9: Acc=0.9418  Sen=0.8973  Spec=0.9863
Patient 10: Acc=0.9382  Sen=0.9551  Spec=0.9213
Patient 11: Acc=0.5676  Sen=0.1887  Spec=0.9465
Patient 12: Acc=0.5035  Sen=0.1831  Spec=0.8239
Patient 13: Acc=0.4231  Sen=0.0385  Spec=0.8077
Patient 14: Acc=0.5976  Sen=0.2241  Spec=0.9710
Patient 15: Acc=0.7381  Sen=0.7143  Spec=0.7619
Patient 16: Acc=0.8660  Sen=0.8557  Spec=0.8763
Patient 17: Acc=0.8883  Sen=0.7864  Spec=0.9903
Patient 18: Acc=0.8846  Sen=0.8077  Spec=0.9615
Patient 19: Acc=0.7368  Sen=0.4947  Spec=0.9789
Patient 20: Acc=0.8692  Sen=0.7692  Spec=0.9692
Pa